# Pregnancy induces transient brain and epigenetic aging followed by postpartum rejuvenation

This Jupyter notebook contains the *R* code that performs the statistical analyses as well as generates the plots and tables presented in the manuscript *Pregnancy induces transient brain and epigenetic aging followed by postpartum rejuvenation*. The notebook is organized in sections, by figures and then tables, that can be runned independently. First, the analyses presented in the main figures are performed, along with the supplementary tables related to them. Then, the analyses for supplementary figures and related supplementary tables are presented. Finally, we show the analyses for the remaining supplementary tables, not asociated to any figure.

For clarity, some repetitive code has been separated in diferent R files contained in the folder "code".

Feather files containing the brainage, epigenetic clocks and demographic data are provided in the folder "data".

# Main Figures

## Figure 1. Datasets

In [ ]:
library(magick)
library(arrow)
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggpubr)

# Basic configuration for plot, defining colorblind-friendly colors and a theme for plots
source("code/basic_plot_configuration.R")


### Cohort Dataset

In [ ]:
# This panel was created separately and saved as a pdf
p_datasets_a <- image_read_pdf("figures/cohort_dataset_subfigure.pdf") |> image_ggplot()

### Dense-sampling Dataset

In [ ]:
dense_bag_data <- read_feather("data/dense_sampling_brainage_data.feather")
dense_epi_data <- read_feather("data/dense_sampling_epigenetic_clocks_data.feather")

dense_data <- merge(dense_bag_data, dense_epi_data, by = c("participant_id", "session"), all = TRUE)
dense_data[dense_data$session == "ses-06-5", "gestation_week"] <- 6.0

In [ ]:
dense_data_plot <- dense_data |>
    mutate(
        Participant = participant_id
        )
dense_data_plot_t1w <- dense_data_plot |>
    filter(brainage_model == "brainAGE") |> 
    select(all_of(c("Participant", "gestation_week", "brainage_gap"))) |>
    drop_na()
dense_data_plot_t1w$data_type <- "T1w"

dense_data_plot_epi <- dense_data_plot |>
    select(all_of(c("Participant", "gestation_week", "PCGrimAge"))) |>
    drop_na()
dense_data_plot_epi$data_type <- "Epigenetics"
dense_data_plot_epi$Participant <- "S01_e"

dense_data_plot <- bind_rows(dense_data_plot_t1w,
                              dense_data_plot_epi)

dense_data_plot$Participant <- factor(dense_data_plot$Participant, 
                                        levels = rev(c("S01", "S01_e", "S02", "S03")))

In [ ]:
p_datasets_b <- dense_data_plot |> ggplot(aes(x=gestation_week, y = Participant, color = Participant)) +
            geom_rug(length = unit(1.0, "npc"), sides = "l", linewidth = 1.2) +
            geom_point(cex = 4, stroke = 1.5, shape = 3) +
            theme_paper +
            scale_color_manual(
            name = "Participant", 
            values = c("S01" = colorblind_colors[2],
                       "S02" = colorblind_colors[3],
                       "S03" = colorblind_colors[4],
                       "S01_e" = colorblind_colors[2]),
            labels = c("S01" = "",
                       "S02" = "S02",
                       "S03" = "S03",
                       "S01_e" = "S01")) +
            geom_vline(xintercept = c(0, 40), linetype = "dashed") +
            scale_x_continuous(breaks = seq(-20, 200, by = 10)) +
            scale_y_discrete(labels = c("S01" = "S01 T1w", "S01_e" = "S01 DNAm",
                                         "S02" = "S02 T1w", "S03" = "S03 T1w")) +
            labs(title = "Dense-Sampling Dataset", x = "Gestation week", y = "Participant/Data") +
            guides(color = guide_legend(reverse = TRUE,
                                        override.aes = list(alpha = c(0,1,1,1)))) +
            theme(legend.position = "top")

In [ ]:
fig_datasets <- ggarrange(p_datasets_a, p_datasets_b,
  nrow = 2,
  heights = c(1.65, 1),
  labels = c("A", "B"),
  font.label = list(size = 20, face = "bold"),
  vjust = 1.5
)

figure_name <- "fig_1_datasets"
figure_path <- paste0("figures/", figure_name)
ggsave(fig_datasets, filename = paste0(figure_path, ".png"), device = "png", width = 15, height = 9, bg = "white")
ggsave(fig_datasets, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 15, height = 9)

## Figure 2: Brain age trajectories (brainageR)

In [ ]:
library(arrow)
library(ggpubr)

# Basic configuration for plot, defining colorblind-friendly colors and a theme for plots
source("code/basic_plot_configuration.R")

# Code containing the function to fit the hierarchical GAM model per brainage model for the cohort dataset
source("code/hgam_fit_per_brainage_model_cohort.R")

# Code containing the function to fit the hierarchical GAM model per brainage model for the dense-sampling dataset
source("code/hgam_fit_per_brainage_model_dense.R")

# Code containing the function to create plots for the hierarchical GAM model fit per brainage model for the cohort dataset
source("code/create_plots_hgam_fit_per_brainage_model_cohort.R")

# Code containing the function to create plots for the hierarchical GAM model fit per brainage model for the dense-sampling dataset
source("code/create_plot_hgam_fit_per_brainage_model_dense.R")

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

hgam_fit_brainageR_cohort <- hgam_fit_per_brainage_model_cohort(cohort_data,
                                                                "brainageR")

#summary(hgam_fit_brainageR_cohort$gam_fit_bam)

In [ ]:
dense_data <- read_feather("data/dense_sampling_brainage_data.feather")

hgam_fit_brainageR_dense <- hgam_fit_per_brainage_model_dense(dense_data,
                                                              "brainageR")

# summary(hgam_fit_brainageR_dense$gam_fit_bam)

In [ ]:
fig_subfig_ylim <- c(-9, 7)
fig_subfig_xlim <- c(-48, 124)

hgam_cohort_plots <- create_plots_hgam_fit_per_brainage_model_cohort(hgam_fit_brainageR_cohort,
                                                        fig_subfig_xlim = fig_subfig_xlim,
                                                        fig_subfig_ylim = fig_subfig_ylim)

In [ ]:
hgam_dense_plot <- create_plot_hgam_fit_per_brainage_model_dense(hgam_fit_brainageR_dense,
                                                    fig_subfig_title = "Gestational mothers (dense-sampled)",
                                                    fig_subfig_xlim = fig_subfig_xlim,
                                                    fig_subfig_ylim = fig_subfig_ylim)

In [ ]:
fig <- ggarrange(hgam_cohort_plots$fig_p1, hgam_dense_plot,
                 hgam_cohort_plots$fig_p2, hgam_cohort_plots$fig_p3,
                 ncol = 2, nrow = 2,
                 labels = c(" A", " B", " C", " D"),
                 font.label = list(size = 18))
figure_name <- "fig_2_brain_age_trajectories_brainageR"
figure_path <- paste0("figures/", figure_name)
ggsave(fig, filename = paste0(figure_path, ".png"), device = "png", width = 15, height = 13, bg = "white")
ggsave(fig, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 15, height = 13)

### Supplementary Table 4: HGAM fit with brainageR for dohort dataset

In [ ]:
# Code containing the functions for formating tables with the results
source("code/formatting_functions.R")

In [ ]:
summ_gam_fit_cohort <- summary(hgam_fit_brainageR_cohort$gam_fit)

hgam_table <- create_hgam_summary_table_docx(model_summary = summ_gam_fit_cohort,
                               stable_substitutions = substitutions_stable_hgam_cohort,
                               ptable_substitutions = substitutions_ptable_hgam_cohort,
                               docx_filename = "tables/smtable_04_hgam_brainageR_cohort.docx")

### Supplementary Table 6: HGAM fit with brainageR for the dense-sampling dataset

In [ ]:
summ_gam_fit_dense <- summary(hgam_fit_brainageR_dense$gam_fit)

hgam_table <- create_hgam_summary_table_docx(model_summary = summ_gam_fit_dense,
                               stable_substitutions = substitutions_stable_hgam_dense,
                               ptable_substitutions = substitutions_ptable_hgam_dense,
                               docx_filename = "tables/smtable_06_hgam_brainageR_dense.docx")

## Figure 3. Brainage and epigenetic clocks correlations

In [ ]:
library(arrow)
library(ggpubr)
library(dplyr)
library(tidyr)
library(stringr)
library(mgcv)
library(gratia)
library(flextable)
library(officer)

# This script contains the function to compute pairwise correlations between the specified metrics in the data frame.
source("code/compute_pairwise_correlations.R")

# This script contains the function to create plots for GAM fit per metric for subject S01
source("code/create_plot_gam_fit_per_metric_s01.R")

# This script contains the function to create correlation plots for the specified metrics for subject S01
source("code/create_correlation_plot_s01.R")

In [ ]:
bag_epi_data <- read_feather("data/brainage_epiclocks_data.feather")

session_means <- bag_epi_data |>
  group_by(session) |>
  summarise(
    n_points = n(),
    mean_age_gap_PCGrimAge = mean(age_gap_PCGrimAge, na.rm = TRUE),
    mean_age_gap_DNAmGrimAge2BasedOnRealAge = mean(age_gap_DNAmGrimAge2BasedOnRealAge, na.rm = TRUE),
    mean_age_gap_PCHorvath1 = mean(age_gap_PCHorvath1, na.rm = TRUE),
    mean_age_gap_PCHorvath2 = mean(age_gap_PCHorvath2, na.rm = TRUE),
    mean_age_gap_PCPhenoAge = mean(age_gap_PCPhenoAge, na.rm = TRUE),
    mean_DunedinPACE = mean(DunedinPACE, na.rm = TRUE),
    .groups = "drop"
  )

bag_epi_data <- bag_epi_data |>
merge(session_means, by = "session") |>
mutate(rep_session = case_when(n_points > 1 ~ session,
                               n_points == 1 ~ NA))
                               
bag_epi_data$rep_session <- as.factor(bag_epi_data$rep_session)

In [ ]:
# Correlations computation
ag_epi_metrics <- c("age_gap_PCHorvath1", "age_gap_PCHorvath2", "age_gap_PCPhenoAge", "age_gap_PCGrimAge", "age_gap_DNAmGrimAge2BasedOnRealAge", "DunedinPACE")
mean_ag_epi_metrics <- c("mean_age_gap_PCHorvath1", "mean_age_gap_PCHorvath2", "mean_age_gap_PCPhenoAge", "mean_age_gap_PCGrimAge", "mean_age_gap_DNAmGrimAge2BasedOnRealAge", "mean_DunedinPACE")
bag_metrics <- c("brainage_gap_pyment", "brainage_gap_brainageR", "brainage_gap_DeepBrainNet", "brainage_gap_brainAGE")

corr_metrics <- c(mean_ag_epi_metrics,
                  bag_metrics)
data_correlations <- bag_epi_data |>
    select(all_of(corr_metrics))

# compute_pairwise_correlations function is defined in the "compute_pairwise_correlations.R" script, which is sourced above. It computes pairwise correlations between the specified metrics in the data frame.
corr_results_df <- compute_pairwise_correlations(data_correlations, corr_metrics)

In [ ]:
# BrainageR
bag_data_brainageR <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "brainage_gap_brainageR", "scanning_site"))) |>
               drop_na()

bag_data_brainageR <- bag_data_brainageR[!duplicated(bag_data_brainageR), ]

bag_gam_fit <- gam(brainage_gap_brainageR ~ s(time_to_parturition_weeks, bs = "tp"),
               data = bag_data_brainageR,
               method = "REML",
               family = "gaussian")

summ_bag_gam_fit <- summary(bag_gam_fit)

bag_gam_fit_sm <- smooth_estimates(bag_gam_fit) |>
    add_confint()

bag_data_brainageR <- bag_data_brainageR |>
    add_partial_residuals(bag_gam_fit)


fig_p1_ylim <- c(-9, 7)
fig_p1_xlim <- c(-48, 128)
fig_p1 <- create_plot_gam_fit_per_metric_s01(bag_data_brainageR,
                                             bag_gam_fit_sm,
                                             fig_subfig_title = "brainageR",
                                             fig_subfig_xlim = fig_p1_xlim,
                                             fig_subfig_ylim = fig_p1_ylim,
                                             fig_subfig_yby = 1,
                                             fig_subfig_ylabel = "Partial residuals of brainage-gap (years)",
                                             fig_subfig_cex = 3,
                                             mark_epi_range = TRUE)

In [ ]:
# PCGrimAge
ag_epi_data_pcgrimage <- bag_epi_data |>
    select(all_of(c("time_to_parturition_weeks", "mean_age_gap_PCGrimAge", "scanning_site"))) |>
    drop_na() 

ag_epi_data_pcgrimage <- ag_epi_data_pcgrimage[!duplicated(ag_epi_data_pcgrimage), ] # Remove duplicates

fit_epi_pcgrimage <- gam(mean_age_gap_PCGrimAge ~ s(time_to_parturition_weeks, bs = "ts") ,
               data = ag_epi_data_pcgrimage,
               method = "REML",
               family = "gaussian") 

summ_fit_epi_pcgrimage <- summary(fit_epi_pcgrimage)

fit_epi_pcgrimage_sm <- smooth_estimates(fit_epi_pcgrimage) |>
    add_confint()

ag_epi_data_pcgrimage <- ag_epi_data_pcgrimage |>
    add_partial_residuals(fit_epi_pcgrimage)

fig_p2_xlim <- c(-48, 56)

fig_p2_1 <- create_plot_gam_fit_per_metric_s01(ag_epi_data_pcgrimage,
                                              fit_epi_pcgrimage_sm,
                                              fig_subfig_title = "PCGrimAge",
                                              fig_subfig_xlim = fig_p2_xlim,
                                              fig_subfig_ylim = fig_p1_ylim,
                                              fig_subfig_yby = 1,
                                              fig_subfig_ylabel = "Partial residuals of age-gap (years)",
                                              fig_subfig_cex = 2,
                                              mark_epi_range = FALSE)

In [ ]:
# DunedinPACE
ag_epi_data_dunedinpace <- bag_epi_data |>
    select(all_of(c("time_to_parturition_weeks", "mean_DunedinPACE", "scanning_site"))) |>
    drop_na()

ag_epi_data_dunedinpace <- ag_epi_data_dunedinpace[!duplicated(ag_epi_data_dunedinpace), ] # Remove duplicates

fit_epi_dunedinpace <- gam(mean_DunedinPACE ~ s(time_to_parturition_weeks),
               data = ag_epi_data_dunedinpace,
               method = "REML",
               family = "gaussian") 

summ_fit_epi_dunedinpace <- summary(fit_epi_dunedinpace)


fit_epi_dunedinpace_sm <- smooth_estimates(fit_epi_dunedinpace) |>
    add_confint()

ag_epi_data_dunedinpace <- ag_epi_data_dunedinpace |>
    add_partial_residuals(fit_epi_dunedinpace)

fig_p2_2 <- create_plot_gam_fit_per_metric_s01(ag_epi_data_dunedinpace,
                                              fit_epi_dunedinpace_sm,
                                              fig_subfig_title = "DunedinPACE",
                                              fig_subfig_xlim = fig_p2_xlim,
                                              fig_subfig_ylim = fig_p1_ylim,
                                              fig_subfig_yby = 0.05,
                                              fig_subfig_ylabel = "Partial effect on aging pace",
                                              fig_subfig_cex = 2,
                                              mark_epi_range = FALSE)

In [ ]:
# Correlation between mean_age_gap_PCGrimAge and brainage_gap_brainageR
bag_brainageR_lims <- c(-10, -2)

get_p2_annot_y <- function(ggfig, perc = 0.9) {
  yrange <- layer_scales(ggfig)$y$range$range
  annot_y <- yrange[1] + (yrange[2] - yrange[1]) * perc
  return(annot_y)
}

pcgrimage_brainager_data <- bag_epi_data |>
    select(all_of(c("mean_age_gap_PCGrimAge", "brainage_gap_brainageR", "scanning_site"))) |>
    drop_na()
pcgrimage_brainager_data <- pcgrimage_brainager_data[!duplicated(pcgrimage_brainager_data), ] # Remove duplicates

# Correlation between mean_age_gap_PCGrimAge and brainage_gap_brainageR
fig_p2_3 <- create_correlation_plot_S01(pcgrimage_brainager_data,
                                        x_metric = "mean_age_gap_PCGrimAge",
                                        y_metric = "brainage_gap_brainageR",
                                        corr_metrics = corr_results_df,
                                        fig_subfig_title = "",
                                        fig_subfig_xlabel = "age-gap (years)",
                                        fig_subfig_ylabel = "brainage-gap (years)",
                                        fig_subfig_xlim = c(0, 20),
                                        fig_subfig_xby = 1,
                                        fig_subfig_ylim = c(-10, -2),
                                        fig_subfig_yby = 1)

In [ ]:
# Correlation between mean_DunedinPACE and brainage_gap_brainageR
dunedinpace_brainager_data <- bag_epi_data |>
    select(all_of(c("mean_DunedinPACE", "brainage_gap_brainageR", "scanning_site"))) |>
    drop_na()
dunedinpace_brainager_data <- dunedinpace_brainager_data[!duplicated(dunedinpace_brainager_data), ] # Remove duplicates

fig_p2_4_cor <- corr_results_df |>
    filter(metric1 == "mean_DunedinPACE" & metric2 == "brainage_gap_brainageR")

fig_p2_4 <- create_correlation_plot_S01(dunedinpace_brainager_data,
                                        x_metric = "mean_DunedinPACE",
                                        y_metric = "brainage_gap_brainageR",
                                        corr_metrics = corr_results_df,
                                        fig_subfig_title = "",
                                        fig_subfig_xlabel = "aging pace",
                                        fig_subfig_ylabel = "brainage-gap (years)",
                                        fig_subfig_xlim = c(0.8, 1.2),
                                        fig_subfig_xby = 0.05,
                                        fig_subfig_ylim = c(-10, -2),
                                        fig_subfig_yby = 1)

In [ ]:
fig <- ggarrange(fig_p1, ggarrange(fig_p2_1, fig_p2_2, fig_p2_3, fig_p2_4,
                        labels = c(" B", " C", " D", " E"),
                        font.label = list(size = 18),
                        ncol = 2,
                        nrow = 2),
          ncol = 2, nrow = 1,
          widths = c(1.3, 1.5),
          labels = c(" A", ""),
          font.label = list(size = 18),
          common.legend = TRUE,
          legend = "bottom"
          )

figure_name <- "fig_3_brainage_epiclocks_trajectories_and_correlations_S01_brainageR"
figure_path <- paste0("figures/", figure_name)
ggsave(fig, filename = paste0(figure_path, ".png"), device = "png", width = 20, height = 10, bg = "white")
ggsave(fig, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 20, height = 10)

### Supplementary Table 18: Brainage epiclock correlation data for participant S01

In [ ]:
corr_substitutions <- c("mean_" = "",
                        "age_gap_" = "Age-gap ",
                        "brainAge-gap " = "Brainage-gap ",
                        "BasedOnRealAge" = "")

corr_results_df$metric1 <- str_replace_all(corr_results_df$metric1, corr_substitutions)
corr_results_df$metric2 <- str_replace_all(corr_results_df$metric2, corr_substitutions)

corr_results_df |> 
  flextable() |>
  width(width = 2, j = 1:2) |>
  width(width = 0.8, j = 3:6) |>
  set_formatter(
    R = function(x) sprintf("%.3f", x),
    df = function(x) sprintf("%d", x),
    p.value = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x)),
    p.FDR = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x))
 
  ) |>
  set_header_labels(metric1 = "Metric 1",
                    metric2 = "Metric 2",
                    R = "R",
                    df = "df",
                    p.value = "p-value",
                    p.FDR = "pFDR"
                    ) |>
    save_as_docx(path = "tables/smtable_18_correlation_results_all.docx")

# Supplementary Figures and Tables

## Supplementary Figure 1. Emmeans plots for LMEMs using BrainageR

In [ ]:
library(arrow)
library(ggpubr)
library(dplyr)
library(tidyr)
library(mgcv)
library(gratia)
library(emmeans)

# Basic configuration for plot, defining colorblind-friendly colors and a theme for plots
source("code/basic_plot_configuration.R")


In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

cohort_data_brainageR <- cohort_data |>
                     select(all_of(c("participant_id", "session", "group_gest", "scanning_site", "brainage_model", "age", "uncorrected_brainage_gap"))) |>
                     filter(brainage_model == 'brainageR') |>
                     mutate(age_centered = age - mean(age)) |>
                     select(-age) |>
                     rename(age = age_centered) |>
                     drop_na()

lme_fit_cohort_brainageR <- gam(uncorrected_brainage_gap ~ group_gest*session +
                                                           s(participant_id, bs='re') + 
                                                           age*scanning_site,
                                data = cohort_data_brainageR,
                                method = "REML", family = "gaussian")

lme_fit_cohort_brainageR_emmeans <- emmeans(lme_fit_cohort_brainageR, ~ session | group_gest)

lme_fit_cohort_brainageR_emmeans_data <- emmip(lme_fit_cohort_brainageR_emmeans, group_gest ~ session, CIs = TRUE, plotit = FALSE) |>
    as_tibble() |>
    mutate(session_time = case_when(
        session == "ses-1" ~ -42,
        session == "ses-2" ~ -20,
        session == "ses-3" ~ -6,
        session == "ses-4" ~ 4,
        session == "ses-5" ~ 25,
        session == "ses-6" ~ 76))


In [ ]:
dense_data <- read_feather("data/dense_sampling_brainage_data.feather")

dense_data <- dense_data |>
  mutate(gestation_postpartum_period = case_when(trimester == "pre"  ~ "pre",
                                 trimester == "first" ~ "first",
                                 trimester == "second" ~ "second",
                                 trimester == "third" ~ "third",
                                 trimester == "post" & gestation_week < 48 ~ "early_post",
                                 trimester == "post" & gestation_week >= 48 & gestation_week < 74 ~ "6m_post",
                                 trimester == "post" & gestation_week >= 74 ~ "year_post")
                                 )

dense_data$gestation_postpartum_period <- factor(dense_data$gestation_postpartum_period,
                                                 levels = c("pre", "first", "second", "third", "early_post", "6m_post", "year_post"))

dense_data_brainageR <- dense_data |> 
  filter(brainage_model == 'brainageR') |>
  select(all_of(c("brainage_gap", "gestation_postpartum_period", "scanning_site", "participant_id", "brainage_model"))) |>
  drop_na()

lme_fit_dense_brainageR <- gam(brainage_gap ~ gestation_postpartum_period + 
                                              scanning_site +
                                              s(participant_id, bs='re'),
                                data = dense_data_brainageR,
                                method = "REML", family = "gaussian")

lme_fit_dense_brainageR_emmeans <- emmeans(lme_fit_dense_brainageR, ~ gestation_postpartum_period)

lme_fit_dense_brainageR_emmeans_data <- emmip(lme_fit_dense_brainageR_emmeans, ~ gestation_postpartum_period , CIs = TRUE, plotit = FALSE) |>
    as_tibble() |>
    mutate(period_time = case_when(
        gestation_postpartum_period == "pre" ~ -42,
        gestation_postpartum_period == "first" ~ -28,
        gestation_postpartum_period == "second" ~ -16,
        gestation_postpartum_period == "third" ~ -4,
        gestation_postpartum_period == "early_post" ~ 4,
        gestation_postpartum_period == "6m_post" ~ 26,
        gestation_postpartum_period == "year_post" ~ 72),
        participant_id = "Global")

per_participant_data_dense_brainageR <- dense_data_brainageR |> group_by(participant_id, gestation_postpartum_period) |>
  summarise(mean_brainage_gap = mean(brainage_gap),
            std_error = sd(brainage_gap) / sqrt(n()),
            .groups = "drop") |>
    mutate(period_time = case_when(
        gestation_postpartum_period == "pre" ~ -42,
        gestation_postpartum_period == "first" ~ -28,
        gestation_postpartum_period == "second" ~ -16,
        gestation_postpartum_period == "third" ~ -4,
        gestation_postpartum_period == "early_post" ~ 4,
        gestation_postpartum_period == "6m_post" ~ 26,
        gestation_postpartum_period == "year_post" ~ 72))

In [ ]:
fig_p1 <- ggplot(lme_fit_cohort_brainageR_emmeans_data,
                 aes(x = session_time, y = yvar, color = group_gest)) +
    geom_point() +
    geom_line() +
    geom_errorbar(aes(ymin = LCL, ymax = UCL), width = 0.2) +
    theme_paper +
    labs(title = "LMEM Cohort Dataset", 
         color = "Group",
         x = "Session",
         y = "Brainage gap (years)") +
    coord_cartesian(xlim = c(-48, 80)) +
    geom_vline(xintercept = c(-40, 0), linetype = "dashed") +
    scale_x_continuous(labels = c( "ses-1", "ses-2", "ses-3", "ses-4", "ses-5", "ses-6"),
                       breaks = c(-42, -20, -6, 4, 25, 76)) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    scale_color_manual(values = c(colorblind_colors[6:8]), labels = c("Gest. mothers", "Non-gest. mothers", "Null. women"))

In [ ]:
fig_p2 <- ggplot(data = lme_fit_dense_brainageR_emmeans_data, 
                 aes(x = period_time, y = yvar, color = participant_id)) +
    geom_point() +
    geom_line() +
    geom_errorbar(aes(ymin = LCL, ymax = UCL), width = 0.2) +
    geom_pointrange(data = per_participant_data_dense_brainageR, 
                    aes(x = period_time,
                        y = mean_brainage_gap,
                        ymin = mean_brainage_gap - 1.96 * std_error,
                        ymax = mean_brainage_gap + 1.96 * std_error,
                        color = participant_id), cex = 0.4, position = position_dodge(width = 0.2)) +
    geom_line(data = per_participant_data_dense_brainageR, 
              aes(x = period_time, y = mean_brainage_gap, group = participant_id, color = participant_id), position = position_dodge(width = 0.2)) +
    theme_paper +
    coord_cartesian(xlim = c(-48, 80)) +
    geom_vline(xintercept = c(-40, 0), linetype = "dashed") +
    scale_x_continuous(labels = c( "Pre-conception", "1st Trimester", "2nd Trimester", "3rd Trimester", "Early Postpartum", "~6M Postpartum", ">~1Y Postpartum"),
                       breaks = c(-42, -28, -16, -4, 4, 26, 72)) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
    labs(title = "LMEM Dense-Sampled Dataset", 
         color = "Participant",
         x = "Gestation/Postpartum period",
         y = "Brainage gap (years)") +
    scale_color_manual(values = c(colorblind_colors[2:4],colorblind_colors[1]), labels = c("S01", "S02", "S03", "Global"))

In [ ]:
fig <- ggarrange(fig_p1, fig_p2,
                  ncol = 2, nrow = 1,
                  labels = c(" A", " B"),
                  font.label = list(size = 18))

figure_name <- "smfig_1_lme_trajectories_brainageR"
figure_path <- paste0("figures/", figure_name)
ggsave(fig, filename = paste0(figure_path, ".png"), device = "png", width = 15, height = 7, bg = "white")
ggsave(fig, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 15, height = 7)


### Supplementary Table 5: LMEM fit for brainage gap using BrainageR on the cohort dataset

In [ ]:
# Code containing the functions for formating tables with the results
source("code/formatting_functions.R")

In [ ]:
# Performing within-group longitudinal comparisons, preserving only those of interest
lme_fit_cohort_brainageR_emmeans_pairs <- lme_fit_cohort_brainageR_emmeans |> 
    pairs(reverse = TRUE)

lme_fit_cohort_brainageR_emmeans_tests <- emmeans::test(lme_fit_cohort_brainageR_emmeans_pairs, by = NULL, adjust = 'none') |> 
      relocate(group_gest, .before = contrast) |>
      filter(str_detect(contrast, paste(c(str_escape("- (ses-1)"), str_escape("- (ses-3)")), collapse = "|"))) |>
      mutate(sesref = str_extract(contrast, "ses-\\d+\\)$")) |> # For ordering the contrasts
      arrange(group_gest, sesref, contrast) |>
      select(-sesref)

lme_fit_cohort_brainageR_emmeans_tests$pFDR <- p.adjust(lme_fit_cohort_brainageR_emmeans_tests$p.value, method = "fdr") 
colnames(lme_fit_cohort_brainageR_emmeans_tests) <- c("Group", "Contrast", "Estimate", "SE",
                                                      "df", "t.ratio", "p.value", "pFDR")

In [ ]:
lme_tables <- create_lme_fit_tables_docx(lme_fit = lme_fit_cohort_brainageR,
                                         em_comparisons = lme_fit_cohort_brainageR_emmeans_tests,
                                         ptable_substitutions = substitutions_ptable_lme_fit_cohort,
                                         stable_substitutions = substitutions_stable_lme_fit_cohort,
                                         emtable_substitutions = substitutions_emtable_test_cohort,
                                         type_dataset = "cohort",
                                         docx_filename = "tables/smtable_05_lme_fit_brainageR_cohort.docx")

### Supplementary Table 7: LMEM fit for brainage gap using BrainageR on the dense-sampling dataset

In [ ]:
# Performing within-group longitudinal comparisons, preserving only those of interest
lme_fit_dense_brainageR_emmeans_pairs <- lme_fit_dense_brainageR_emmeans |>
      pairs(reverse = TRUE)

lme_fit_dense_brainageR_emmeans_tests <- emmeans::test(lme_fit_dense_brainageR_emmeans_pairs, by = NULL, adjust = 'none') 

lme_fit_dense_brainageR_emmeans_tests_pre <- lme_fit_dense_brainageR_emmeans_tests |>
      filter(str_detect(contrast, "pre"))

lme_fit_dense_brainageR_emmeans_tests_third <- lme_fit_dense_brainageR_emmeans_tests |>
      filter(str_detect(contrast, str_escape("- third")))

lme_fit_dense_brainageR_emmeans_tests <- rbind(lme_fit_dense_brainageR_emmeans_tests_pre, lme_fit_dense_brainageR_emmeans_tests_third) 
    
lme_fit_dense_brainageR_emmeans_tests$pFDR <- p.adjust(lme_fit_dense_brainageR_emmeans_tests$p.value, method = "fdr")
colnames(lme_fit_dense_brainageR_emmeans_tests) <- c("Contrast", "Estimate", "SE",
                                                      "df", "t.ratio", "p.value", "pFDR")

In [ ]:
lme_tables <- create_lme_fit_tables_docx(
    lme_fit = lme_fit_dense_brainageR,
    em_comparisons = lme_fit_dense_brainageR_emmeans_tests,
    ptable_substitutions = substitutions_ptable_lme_fit_dense,
    stable_substitutions = substitutions_stable_lme_fit_dense,
    emtable_substitutions = substitutions_emtable_test_dense,
    type_dataset = "dense",
    docx_filename = "tables/smtable_07_lme_fit_brainageR_dense.docx"
)
                                       

## Supplementary Figure 2. Brainage gap trajectories for Pyment, DeepBrainNet and BrainAGE algorithms for cohort dataset

In [ ]:
library(arrow)
library(ggpubr)

source("code/hgam_fit_per_brainage_model_cohort.R")
source("code/create_plots_hgam_fit_per_brainage_model_cohort.R")
source("code/formatting_functions.R")

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

In [ ]:
# Pyment 
hgam_fit_pyment <- hgam_fit_per_brainage_model_cohort(cohort_data, "pyment")
figs_pyment <- create_plots_hgam_fit_per_brainage_model_cohort(hgam_fit_pyment,
                                                               fig_subfig_xlim = c(-48, 108),
                                                               fig_subfig_ylim = c(-10, 10))

In [ ]:
# DeepBrainNet
hgam_fit_deepbrainnet <- hgam_fit_per_brainage_model_cohort(cohort_data, "DeepBrainNet")
figs_deepbrainnet <- create_plots_hgam_fit_per_brainage_model_cohort(hgam_fit_deepbrainnet,
                                                                     fig_subfig_xlim = c(-48, 108),
                                                                     fig_subfig_ylim = c(-10, 10))

In [ ]:
# BrainAge
hgam_fit_brainage <- hgam_fit_per_brainage_model_cohort(cohort_data, "brainAGE")
figs_brainage <- create_plots_hgam_fit_per_brainage_model_cohort(hgam_fit_brainage,
                                                                 fig_subfig_xlim = c(-48, 108),
                                                                 fig_subfig_ylim = c(-10, 10))

In [ ]:
fig <- ggarrange(figs_pyment$fig_p1, figs_pyment$fig_p2, figs_pyment$fig_p3,
          figs_deepbrainnet$fig_p1, figs_deepbrainnet$fig_p2, figs_deepbrainnet$fig_p3,
          figs_brainage$fig_p1, figs_brainage$fig_p2, figs_brainage$fig_p3,
          ncol = 3, nrow = 3,
          labels = c("A", "B", "C", "D", "E", "F", "G", "H", "I"),
          font.label = list(size = 18),
          common.legend = TRUE, legend = "bottom") +
theme(plot.margin = margin(t = 0, r = 0, b = 0, l = 50)) +
annotate(geom = "text", x = -0.02, y = 0.84, label = "Pyment", size = 14, angle = 90) +
annotate(geom = "text", x = -0.02, y = 0.51, label = "DeepBrainNet", size = 14, angle = 90) +
annotate(geom = "text", x = -0.02, y = 0.20, label = "brainAGE", size = 14, angle = 90)

figure_name <- "smfig_2_brainage_trajectories_additional_brainage_models_cohort"
figure_path <- paste0("figures/", figure_name)
ggsave(fig, filename = paste0(figure_path, ".png"), device = "png", width = 20, height = 20, bg = "white")
ggsave(fig, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 20, height = 20)

### Supplementary Table 9: HGAM fit for Pyment in cohort dataset

In [ ]:
summ_hgam_fit_pyment_cohort <- summary(hgam_fit_pyment$gam_fit)

hgam_table <- create_hgam_summary_table_docx(model_summary = summ_hgam_fit_pyment_cohort,
                               stable_substitutions = substitutions_stable_hgam_cohort,
                               ptable_substitutions = substitutions_ptable_hgam_cohort,
                               docx_filename = "tables/smtable_09_hgam_pyment_cohort.docx")

### Supplementary Table 10: HGAM fit for DeepBrainNet in cohort dataset

In [ ]:
summ_gam_fit_deepbrainnet_cohort <- summary(hgam_fit_deepbrainnet$gam_fit)

hgam_table <- create_hgam_summary_table_docx(model_summary = summ_gam_fit_deepbrainnet_cohort,
                                stable_substitutions = substitutions_stable_hgam_cohort,
                                ptable_substitutions = substitutions_ptable_hgam_cohort,
                                docx_filename = "tables/smtable_10_hgam_DeepBrainNet_cohort.docx")

### Supplementary Table 11: HGAM fit for brainAGE in cohort dataset

In [ ]:
summ_gam_fit_brainage_cohort <- summary(hgam_fit_brainage$gam_fit)

hgam_table <- create_hgam_summary_table_docx(model_summary = summ_gam_fit_brainage_cohort,
                               stable_substitutions = substitutions_stable_hgam_cohort,
                               ptable_substitutions = substitutions_ptable_hgam_cohort,
                               docx_filename = "tables/smtable_11_hgam_brainAGE_cohort.docx")

## Supplementary Figure 3. Brainage gap trajectories for Pyment, DeepBrainNet and BrainAGE algorithms for dense-sampling dataset

In [ ]:
library(arrow)
library(ggpubr)
library(tibble)
library(flextable)
library(officer)

# Contains function for fitting HGAMs to the dense-sampled dataset
source("code/hgam_fit_per_brainage_model_dense.R")
# Contains function for creating plot of HGAM fits to the dense-sampled dataset
source("code/create_plot_hgam_fit_per_brainage_model_dense.R")
# Contains functions for formatting tables with the results
source("code/formatting_functions.R")

In [ ]:
dense_data <- read_feather("data/dense_sampling_brainage_data.feather")

In [ ]:
# Pyment
hgam_fit_pyment <- hgam_fit_per_brainage_model_dense(dense_data, "pyment")
p_pyment <- create_plot_hgam_fit_per_brainage_model_dense(hgam_fit_pyment, "Pyment")

In [ ]:
# DeepBrainNet
hgam_fit_deepbrainnet <- hgam_fit_per_brainage_model_dense(dense_data, "DeepBrainNet")
p_deepbrainnet <- create_plot_hgam_fit_per_brainage_model_dense(hgam_fit_deepbrainnet, "DeepBrainNet")

In [ ]:
# BrainAGE
hgam_fit_brainAGE <- hgam_fit_per_brainage_model_dense(dense_data, "brainAGE")
p_brainAGE <- create_plot_hgam_fit_per_brainage_model_dense(hgam_fit_brainAGE, "brainAGE")

In [ ]:
fig <- ggarrange(p_pyment, p_deepbrainnet, p_brainAGE,
                 ncol = 3, nrow = 1,
                 labels = c(" A", " B", " C"),
                 font.label = list(size = 18),
                 common.legend = TRUE, legend = "bottom")  

figure_name <- "smfig_3_brainage_trajectories_additional_brainage_models_dense"
figure_path <- paste0("figures/", figure_name)
ggsave(fig, filename = paste0(figure_path, ".png"), device = "png", width = 20, height = 7, bg = "white")
ggsave(fig, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 20, height = 7)

### Supplementary Table 15: HGAM fit for the pyment, DeepBrainNet and brainAGE algorithms in the dense-sampling dataset

In [ ]:
summ_gam_fit_pyment_dense <- summary(hgam_fit_pyment$gam_fit)
summ_gam_fit_deepbrainnet_dense <- summary(hgam_fit_deepbrainnet$gam_fit)
summ_gam_fit_brainAGE_dense <- summary(hgam_fit_brainAGE$gam_fit)

# Join HGAM ptables
ptable_pyment <- data.frame(summ_gam_fit_pyment_dense$p.table) |> rownames_to_column(var = "Coefficient")
ptable_deepbrainnet <- data.frame(summ_gam_fit_deepbrainnet_dense$p.table) |> rownames_to_column(var = "Coefficient")
ptable_brainAGE <- data.frame(summ_gam_fit_brainAGE_dense$p.table) |> rownames_to_column(var = "Coefficient")

ptable_pyment$brainage_model <- "pyment"
ptable_deepbrainnet$brainage_model <- "DeepBrainNet"
ptable_brainAGE$brainage_model <- "brainAGE"


joined_ptables <- bind_rows(ptable_pyment, ptable_deepbrainnet, ptable_brainAGE) |>
                                 relocate(brainage_model, .before = Coefficient)


colnames(joined_ptables)[1] <- "Brainage Model"
colnames(joined_ptables)[4] <- "Std.Error"
colnames(joined_ptables)[6] <- "p.value"

joined_ptables_formated <- joined_ptables |>
  flextable() |> 
  width(width = 1.5, j = 1) |>
  width(width = 3, j = 2) |>
  width(width = 1, j = 3:6) |> 
  set_formatter(
    "Estimate" = function(x) sprintf("%.2f", x),
    "Std.Error" = function(x) sprintf("%.2f", x),
    "t.value" = function(x) sprintf("%.2f", x),
    "p.value" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x))
  )

# Join HGAM stables
stable_pyment <- data.frame(summ_gam_fit_pyment_dense$s.table) |> rownames_to_column(var = "Smooth Term")
stable_deepbrainnet <- data.frame(summ_gam_fit_deepbrainnet_dense$s.table) |> rownames_to_column(var = "Smooth Term")
stable_brainAGE <- data.frame(summ_gam_fit_brainAGE_dense$s.table) |> rownames_to_column(var = "Smooth Term") 

stable_pyment$brainage_model <- "pyment"
stable_deepbrainnet$brainage_model <- "DeepBrainNet"
stable_brainAGE$brainage_model <- "brainAGE"

joined_stables <- bind_rows(stable_pyment, stable_deepbrainnet, stable_brainAGE) |>
                                 relocate(brainage_model)

colnames(joined_stables)[1] <- "Brainage Model"

joined_stables_formated <- joined_stables |> 
  flextable() |> 
  width(width = 1.5, j = 1) |>
  width(width = 3, j = 2) |>
  width(width = 1, j = 3:6) |> 
  set_formatter(
    "edf" = function(x) sprintf("%.2f", x),
    "Ref.df" = function(x) sprintf("%.2f", x),
    "F" = function(x) sprintf("%.2f", x),
    "p.value" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x))
  )

# Join HGAM Fit quality metrics
joined_hgam_fit_quality_metrics <- data.frame(
    Brainage_Model = c("pyment", "DeepBrainNet", "brainAGE"),
    r.sq = c(summ_gam_fit_pyment_dense$r.sq, summ_gam_fit_deepbrainnet_dense$r.sq, summ_gam_fit_brainAGE_dense$r.sq),
    dev.expl = c(summ_gam_fit_pyment_dense$dev.expl, summ_gam_fit_deepbrainnet_dense$dev.expl, summ_gam_fit_brainAGE_dense$dev.expl),
    n = c(summ_gam_fit_pyment_dense$n, summ_gam_fit_deepbrainnet_dense$n, summ_gam_fit_brainAGE_dense$n)
)

# Create tables
joined_hgam_fit_quality_metrics_formated <- joined_hgam_fit_quality_metrics |> 
    flextable() |> 
    width(width = 1.5, j = 1) |>
    width(width = 1.5, j = 2:4) |> 
    set_formatter(
        "r.sq" = function(x) sprintf("%.3f", x),
        "dev.expl" = function(x) sprintf("%.3f", x),
        "n" = function(x) sprintf("%d", x)
    ) |> 
    set_header_labels(
        Brainage_Model = "Brainage Model",
        r.sq = "R-squared",
        dev.expl = "Deviance Explained",
        n = "N"
    )

joined_tables_hgam_docx <- read_docx() |>
    body_add_par("Table with HGAM parametric coefficients") |>
    body_add_flextable(joined_ptables_formated) |>
    body_add_par("Table with HGAM smooth terms") |>
    body_add_flextable(joined_stables_formated) |>
    body_add_par("HGAM Fit quality metrics") |>
    body_add_flextable(joined_hgam_fit_quality_metrics_formated) |>
    print(target = "tables/smtable_15_hgam_fit_other_brainage_algorithms_joined_dense.docx")

## Supplementary Figure 4. Additional epigenetic clock trajectories and correlations with brainageR 

In [ ]:
library(arrow)
library(ggpubr)
library(dplyr)
library(tidyr)
library(mgcv)
library(gratia)
library(flextable)
library(officer)

# This script contains the function to compute pairwise correlations between the specified metrics in the data frame.
source("code/compute_pairwise_correlations.R")

# This script contains the function to create plots for GAM fit per metric for subject S01
source("code/create_plot_gam_fit_per_metric_s01.R")

# This script contains the function to create correlation plots for the specified metrics for subject S01
source("code/create_correlation_plot_s01.R")

# This script contains the function to format tables with the results
source("code/formatting_functions.R")

In [ ]:
bag_epi_data <- read_feather("data/brainage_epiclocks_data.feather")

session_means <- bag_epi_data |>
  group_by(session) |>
  summarise(
    n_points = n(),
    mean_age_gap_PCGrimAge = mean(age_gap_PCGrimAge, na.rm = TRUE),
    mean_age_gap_DNAmGrimAge2BasedOnRealAge = mean(age_gap_DNAmGrimAge2BasedOnRealAge, na.rm = TRUE),
    mean_age_gap_PCHorvath1 = mean(age_gap_PCHorvath1, na.rm = TRUE),
    mean_age_gap_PCHorvath2 = mean(age_gap_PCHorvath2, na.rm = TRUE),
    mean_age_gap_PCPhenoAge = mean(age_gap_PCPhenoAge, na.rm = TRUE),
    mean_DunedinPACE = mean(DunedinPACE, na.rm = TRUE),
    .groups = "drop"
  )  

bag_epi_data <- bag_epi_data |> 
  merge(session_means, by = "session") |> 
  mutate(rep_session = case_when(n_points > 1 ~ session,
                                 n_points == 1 ~ NA)) 
bag_epi_data$rep_session <- as.factor(bag_epi_data$rep_session)

# Correlations computation
ag_epi_metrics <- c("age_gap_PCHorvath1", "age_gap_PCHorvath2", "age_gap_PCPhenoAge", "age_gap_PCGrimAge", "age_gap_DNAmGrimAge2BasedOnRealAge", "DunedinPACE")
mean_ag_epi_metrics <- c("mean_age_gap_PCHorvath1", "mean_age_gap_PCHorvath2", "mean_age_gap_PCPhenoAge", "mean_age_gap_PCGrimAge", "mean_age_gap_DNAmGrimAge2BasedOnRealAge", "mean_DunedinPACE")
bag_metrics <- c("brainage_gap_pyment", "brainage_gap_brainageR", "brainage_gap_DeepBrainNet", "brainage_gap_brainAGE")

corr_metrics <- c(mean_ag_epi_metrics, 
                  bag_metrics)
data_correlations <- bag_epi_data |>
    select(all_of(corr_metrics))

# compute_pairwise_correlations function is defined in the "compute_pairwise_correlations.R" script, which is sourced above. It computes pairwise correlations between the specified metrics in the data frame.
corr_results_df <- compute_pairwise_correlations(data_correlations, corr_metrics)

In [ ]:
fig_subplot_ylim <- c(-10, 7)
fig_subplot_xlim <- c(-40, 56)

# PCHorvath1
data_ag_epi_pchorvath1 <- bag_epi_data |>
    select(all_of(c("time_to_parturition_weeks", "mean_age_gap_PCHorvath1", "scanning_site"))) |>
    drop_na()

data_ag_epi_pchorvath1 <- data_ag_epi_pchorvath1[!duplicated(data_ag_epi_pchorvath1), ]

fit_epi_pchorvath1 <- gam(mean_age_gap_PCHorvath1 ~ s(time_to_parturition_weeks),
               data = data_ag_epi_pchorvath1,
               method = "REML",
               family = "gaussian")

fit_epi_pchorvath1_sm <- smooth_estimates(fit_epi_pchorvath1) |>
    add_confint()

data_ag_epi_pchorvath1 <- data_ag_epi_pchorvath1 |>
    add_partial_residuals(fit_epi_pchorvath1)

fig_p1_1 <- create_plot_gam_fit_per_metric_s01(data_ag_epi_pchorvath1,
                                              fit_epi_pchorvath1_sm,
                                              fig_subfig_title = "PCHorvath1",
                                              fig_subfig_xlim = fig_subplot_xlim,
                                              fig_subfig_ylim = fig_subplot_ylim,
                                              fig_subfig_yby = 1,
                                              fig_subfig_ylabel = "Partial effect on age-gap (years)",
                                              fig_subfig_cex = 2,
                                              mark_epi_range = FALSE)

In [ ]:
#PCHorvath2
data_ag_epi_pchorvath2 <- bag_epi_data |>
    select(all_of(c("time_to_parturition_weeks", "mean_age_gap_PCHorvath2", "scanning_site"))) |>
    drop_na()

data_ag_epi_pchorvath2 <- data_ag_epi_pchorvath2[!duplicated(data_ag_epi_pchorvath2), ]

fit_epi_pchorvath2 <- gam(mean_age_gap_PCHorvath2 ~ s(time_to_parturition_weeks),
               data = data_ag_epi_pchorvath2,
               method = "REML",
               family = "gaussian")
fit_epi_pchorvath2_sm <- smooth_estimates(fit_epi_pchorvath2) |>
    add_confint()
data_ag_epi_pchorvath2 <- data_ag_epi_pchorvath2 |> 
    add_partial_residuals(fit_epi_pchorvath2)

fig_p1_2 <- create_plot_gam_fit_per_metric_s01(data_ag_epi_pchorvath2,
                                              fit_epi_pchorvath2_sm,
                                              fig_subfig_title = "PCHorvath2",
                                              fig_subfig_xlim = fig_subplot_xlim,
                                              fig_subfig_ylim = fig_subplot_ylim,
                                              fig_subfig_yby = 1,
                                              fig_subfig_ylabel = "Partial effect on age-gap (years)",
                                              fig_subfig_cex = 2,
                                              mark_epi_range = FALSE)

In [ ]:
#PCPhenoAge
data_ag_epi_pcphenoage <- bag_epi_data |>
    select(all_of(c("time_to_parturition_weeks", "mean_age_gap_PCPhenoAge", "scanning_site"))) |>
    drop_na()

data_ag_epi_pcphenoage <- data_ag_epi_pcphenoage[!duplicated(data_ag_epi_pcphenoage), ]

fit_epi_pcphenoage <- gam(mean_age_gap_PCPhenoAge ~ s(time_to_parturition_weeks),
               data = data_ag_epi_pcphenoage,
               method = "REML",
               family = "gaussian")
fit_epi_pcphenoage_sm <- smooth_estimates(fit_epi_pcphenoage) |>
    add_confint()
data_ag_epi_pcphenoage <- data_ag_epi_pcphenoage |>
    add_partial_residuals(fit_epi_pcphenoage)
fig_p1_3 <- create_plot_gam_fit_per_metric_s01(data_ag_epi_pcphenoage,
                                              fit_epi_pcphenoage_sm,
                                              fig_subfig_title = "PCPhenoAge",
                                              fig_subfig_xlim = fig_subplot_xlim,
                                              fig_subfig_ylim = fig_subplot_ylim,
                                              fig_subfig_yby = 1,
                                              fig_subfig_ylabel = "Partial effect on age-gap (years)",
                                              fig_subfig_cex = 2,
                                              mark_epi_range = FALSE)

In [ ]:
#DNAmGrimAge2BasedOnRealAge
data_ag_epi_pcgrimage2 <- bag_epi_data |>
    select(all_of(c("time_to_parturition_weeks", "mean_age_gap_DNAmGrimAge2BasedOnRealAge", "scanning_site"))) |>
    drop_na()

data_ag_epi_pcgrimage2 <- data_ag_epi_pcgrimage2[!duplicated(data_ag_epi_pcgrimage2), ]

fit_epi_pcgrimage2 <- gam(mean_age_gap_DNAmGrimAge2BasedOnRealAge ~ s(time_to_parturition_weeks),
               data = data_ag_epi_pcgrimage2,
               method = "REML",
               family = "gaussian")
fit_epi_pcgrimage2_sm <- smooth_estimates(fit_epi_pcgrimage2) |>
    add_confint()
data_ag_epi_pcgrimage2 <- data_ag_epi_pcgrimage2 |>
    add_partial_residuals(fit_epi_pcgrimage2)
fig_p1_4 <- create_plot_gam_fit_per_metric_s01(data_ag_epi_pcgrimage2,
                                              fit_epi_pcgrimage2_sm,
                                              fig_subfig_title = "DNAmGrimAge2",
                                              fig_subfig_xlim = fig_subplot_xlim,
                                              fig_subfig_ylim = fig_subplot_ylim,
                                              fig_subfig_yby = 1,
                                              fig_subfig_ylabel = "Partial effect on age-gap (years)",
                                              fig_subfig_cex = 2,
                                              mark_epi_range = FALSE)

In [ ]:
bag_brainageR_lims <- c(-10, -2)

# Correlation between mean_age_gap_PCHorvath1 and brainage_gap_brainageR
ag_h1_lims <- c(-9, -3)

pchorvath1_brainager_gap <- bag_epi_data |>
    select(all_of(c("mean_age_gap_PCHorvath1", "brainage_gap_brainageR", "scanning_site"))) |>
    drop_na()
pchorvath1_brainager_gap <- pchorvath1_brainager_gap[!duplicated(pchorvath1_brainager_gap), ]

fig_p2_1 <- create_correlation_plot_S01(pchorvath1_brainager_gap,
                                        x_metric = "mean_age_gap_PCHorvath1",
                                        y_metric = "brainage_gap_brainageR",
                                        corr_metrics = corr_results_df,
                                        fig_subfig_title = "PCHorvath1 vs brainageR",
                                        fig_subfig_xlabel = "age-gap (years)",
                                        fig_subfig_ylabel = "brainage-gap (years)",
                                        fig_subfig_xlim = ag_h1_lims,
                                        fig_subfig_xby = 1,
                                        fig_subfig_ylim = bag_brainageR_lims,
                                        fig_subfig_yby = 1)

In [ ]:
# Correlation between mean_age_gap_PCHorvath2 and brainage_gap_brainageR
ag_h2_lims <- c(-11, -5)

pchorvath2_brainager_gap <- bag_epi_data |>
    select(all_of(c("mean_age_gap_PCHorvath2", "brainage_gap_brainageR", "scanning_site"))) |>
    drop_na()
pchorvath2_brainager_gap <- pchorvath2_brainager_gap[!duplicated(pchorvath2_brainager_gap), ]

fig_p2_2 <- create_correlation_plot_S01(pchorvath2_brainager_gap,
                                        x_metric = "mean_age_gap_PCHorvath2",
                                        y_metric = "brainage_gap_brainageR",
                                        corr_metrics = corr_results_df,
                                        fig_subfig_title = "PCHorvath2 vs brainageR",
                                        fig_subfig_xlabel = "age-gap (years)",
                                        fig_subfig_ylabel = "brainage-gap (years)",
                                        fig_subfig_xlim = ag_h2_lims,
                                        fig_subfig_xby = 1,
                                        fig_subfig_ylim = bag_brainageR_lims,
                                        fig_subfig_yby = 1)



In [ ]:
# Correlation between mean_age_gap_PCPhenoAge and brainage_gap_brainageR
ag_pa_lims <- c(-9, 6)

pcphenoage_brainager_gap <- bag_epi_data |>
    select(all_of(c("mean_age_gap_PCPhenoAge", "brainage_gap_brainageR", "scanning_site"))) |>
    drop_na()
pcphenoage_brainager_gap <- pcphenoage_brainager_gap[!duplicated(pcphenoage_brainager_gap), ]

fig_p2_3 <- create_correlation_plot_S01(pcphenoage_brainager_gap,
                                        x_metric = "mean_age_gap_PCPhenoAge",
                                        y_metric = "brainage_gap_brainageR",
                                        corr_metrics = corr_results_df,
                                        fig_subfig_title = "PCPhenoAge vs brainageR",
                                        fig_subfig_xlabel = "age-gap (years)",
                                        fig_subfig_ylabel = "brainage-gap (years)",
                                        fig_subfig_xlim = ag_pa_lims,
                                        fig_subfig_xby = 1,
                                        fig_subfig_ylim = bag_brainageR_lims,
                                        fig_subfig_yby = 1)

In [ ]:
# Correlation between mean_age_gap_DNAmGrimAge2BasedOnRealAge and brainage_gap_brainageR
ag_ga2_lims <- c(18, 31)

pcgrimage2_brainager_gap <- bag_epi_data |>
    select(all_of(c("mean_age_gap_DNAmGrimAge2BasedOnRealAge", "brainage_gap_brainageR", "scanning_site"))) |>
    drop_na()
pcgrimage2_brainager_gap <- pcgrimage2_brainager_gap[!duplicated(pcgrimage2_brainager_gap), ]

fig_p2_4 <- create_correlation_plot_S01(pcgrimage2_brainager_gap,
                                        x_metric = "mean_age_gap_DNAmGrimAge2BasedOnRealAge",
                                        y_metric = "brainage_gap_brainageR",
                                        corr_metrics = corr_results_df,
                                        fig_subfig_title = "DNAmGrimAge2 vs brainageR",
                                        fig_subfig_xlabel = "age-gap (years)",
                                        fig_subfig_ylabel = "brainage-gap (years)",
                                        fig_subfig_xlim = ag_ga2_lims,
                                        fig_subfig_xby = 1,
                                        fig_subfig_ylim = bag_brainageR_lims,
                                        fig_subfig_yby = 1)


In [ ]:
fig <- ggarrange(fig_p1_1, fig_p1_2, fig_p1_3, fig_p1_4, fig_p2_1, fig_p2_2, fig_p2_3, fig_p2_4,
          labels = c(" A", " B", " C", " D", " E", " F", " G", " H"),
          font.label = list(size = 18),
          ncol = 4,
          nrow = 2,
          common.legend = TRUE,
          legend = "bottom") +
          theme(plot.margin = margin(t = 0, r = 15, b = 0, l = 0)) -> sfig3

# Save the figures
figure_name <- "smfig_4_epigenetic_clocks_trajectories_and_correlations_additional_algorithms_S01_brainageR"
figure_path <- paste0("figures/", figure_name)
ggsave(fig, filename = paste0(figure_path, ".png"), device = "png", width = 20, height = 12, bg = "white")
ggsave(fig, filename = paste0(figure_path, ".pdf"), device = "pdf", width = 20, height = 12)

### Supplementary Table 17: GAM fitted metrics for S01 data

In [ ]:
gam_fitted_metrics_s01 <- data.frame(metric = character(),
                                     smooth.edf = numeric(),
                                     smooth.ref.df = numeric(),
                                     smooth.F = numeric(),
                                     smooth.p = numeric(),
                                     r_sq = numeric(),
                                     n = integer()
                                     )

In [ ]:
# brainage [brainageR]
data_bag_brainageR <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "brainage_gap_brainageR", "scanning_site"))) |>
               drop_na()

data_bag_brainageR <- data_bag_brainageR[!duplicated(data_bag_brainageR), ] 

fit_epi_brainageR <- gam(brainage_gap_brainageR ~ s(time_to_parturition_weeks, bs = "tp"),
               data = data_bag_brainageR,
               method = "REML",
               family = "gaussian")

summ_fit_epi_brainageR <- summary(fit_epi_brainageR)

gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Brainage-gap [brainageR]",
            smooth.edf = summ_fit_epi_brainageR$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_brainageR$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_brainageR$s.table[1, "F"],
            smooth.p = summ_fit_epi_brainageR$s.table[1, "p-value"],
            r_sq = summ_fit_epi_brainageR$r.sq,
            n = summ_fit_epi_brainageR$n
            )

In [ ]:
# brainage [pyment]
data_bag_pyment <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "brainage_gap_pyment", "scanning_site"))) |>
               drop_na()

data_bag_pyment <- data_bag_pyment[!duplicated(data_bag_pyment), ] 

fit_epi_pyment <- gam(brainage_gap_pyment ~ s(time_to_parturition_weeks, bs = "tp"),
               data = data_bag_pyment,
               method = "REML",
               family = "gaussian")

summ_fit_epi_pyment <- summary(fit_epi_pyment)

gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Brainage-gap [pyment]",
            smooth.edf = summ_fit_epi_pyment$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_pyment$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_pyment$s.table[1, "F"],
            smooth.p = summ_fit_epi_pyment$s.table[1, "p-value"],
            r_sq = summ_fit_epi_pyment$r.sq,
            n = summ_fit_epi_pyment$n
            )

In [ ]:
# brainage [DeepBrainNet]
data_bag_DeepBrainNet <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "brainage_gap_DeepBrainNet", "scanning_site"))) |>
               drop_na()

data_bag_DeepBrainNet <- data_bag_DeepBrainNet[!duplicated(data_bag_DeepBrainNet), ] 

fit_epi_DeepBrainNet <- gam(brainage_gap_DeepBrainNet ~ s(time_to_parturition_weeks, bs = "tp"),
               data = data_bag_DeepBrainNet,
               method = "REML",
               family = "gaussian")

summ_fit_epi_DeepBrainNet <- summary(fit_epi_DeepBrainNet)

gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Brainage-gap [DeepBrainNet]",
            smooth.edf = summ_fit_epi_DeepBrainNet$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_DeepBrainNet$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_DeepBrainNet$s.table[1, "F"],
            smooth.p = summ_fit_epi_DeepBrainNet$s.table[1, "p-value"],
            r_sq = summ_fit_epi_DeepBrainNet$r.sq,
            n = summ_fit_epi_DeepBrainNet$n
            )

In [ ]:
# brainage [brainAGE]
data_bag_brainAGE <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "brainage_gap_brainAGE", "scanning_site"))) |>
               drop_na()

data_bag_brainAGE <- data_bag_brainAGE[!duplicated(data_bag_brainAGE), ] 

fit_epi_brainAGE <- gam(brainage_gap_brainAGE ~ s(time_to_parturition_weeks, bs = "tp"),
               data = data_bag_brainAGE,
               method = "REML",
               family = "gaussian")

summ_fit_epi_brainAGE <- summary(fit_epi_brainAGE)

gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Brainage-gap [brainAGE]",
            smooth.edf = summ_fit_epi_brainAGE$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_brainAGE$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_brainAGE$s.table[1, "F"],
            smooth.p = summ_fit_epi_brainAGE$s.table[1, "p-value"],
            r_sq = summ_fit_epi_brainAGE$r.sq,
            n = summ_fit_epi_brainAGE$n
            )

In [ ]:
# Epigenetic clock PCHorvath1
summ_fit_epi_pchorvath1 <- summary(fit_epi_pchorvath1)

gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Age-gap [PCHorvath1]",
            smooth.edf = summ_fit_epi_pchorvath1$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_pchorvath1$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_pchorvath1$s.table[1, "F"],
            smooth.p = summ_fit_epi_pchorvath1$s.table[1, "p-value"],
            r_sq = summ_fit_epi_pchorvath1$r.sq,
            n = summ_fit_epi_pchorvath1$n
            )

In [ ]:
# Epigenetic clock PCHorvath2
summ_fit_epi_pchorvath2 <- summary(fit_epi_pchorvath2)
gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Age-gap [PCHorvath2]",
            smooth.edf = summ_fit_epi_pchorvath2$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_pchorvath2$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_pchorvath2$s.table[1, "F"],
            smooth.p = summ_fit_epi_pchorvath2$s.table[1, "p-value"],
            r_sq = summ_fit_epi_pchorvath2$r.sq,
            n = summ_fit_epi_pchorvath2$n
            )

In [ ]:
# Epigenetic clock PCPhenoAge
summ_fit_epi_pcphenoage <- summary(fit_epi_pcphenoage)
gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Age-gap [PCPhenoAge]",
            smooth.edf = summ_fit_epi_pcphenoage$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_pcphenoage$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_pcphenoage$s.table[1, "F"],
            smooth.p = summ_fit_epi_pcphenoage$s.table[1, "p-value"],
            r_sq = summ_fit_epi_pcphenoage$r.sq,
            n = summ_fit_epi_pcphenoage$n
            )

In [ ]:
# Epigenetic clock PCGrimAge
data_epi_pcgrimage <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "mean_age_gap_PCGrimAge", "scanning_site"))) |>
               drop_na()

data_epi_pcgrimage <- data_epi_pcgrimage[!duplicated(data_epi_pcgrimage), ]
fit_epi_pcgrimage <- gam(mean_age_gap_PCGrimAge ~ s(time_to_parturition_weeks, bs = "tp"),
               data = data_epi_pcgrimage,
               method = "REML",
               family = "gaussian")
summ_fit_epi_pcgrimage <- summary(fit_epi_pcgrimage)
gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Age-gap [PCGrimAge]",
            smooth.edf = summ_fit_epi_pcgrimage$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_pcgrimage$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_pcgrimage$s.table[1, "F"],
            smooth.p = summ_fit_epi_pcgrimage$s.table[1, "p-value"],
            r_sq = summ_fit_epi_pcgrimage$r.sq,
            n = summ_fit_epi_pcgrimage$n
            )

In [ ]:
# Epigenetic clock PCGrimAge2BasedOnRealAge
summ_fit_epi_pcgrimage2 <- summary(fit_epi_pcgrimage2)
gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "Age-gap [DNAmGrimAge2]",
            smooth.edf = summ_fit_epi_pcgrimage2$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_pcgrimage2$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_pcgrimage2$s.table[1, "F"],
            smooth.p = summ_fit_epi_pcgrimage2$s.table[1, "p-value"],
            r_sq = summ_fit_epi_pcgrimage2$r.sq,
            n = summ_fit_epi_pcgrimage2$n
            )

In [ ]:
# Epigenetic clock DunedinPACE
data_epi_dunedinpace <- bag_epi_data |>
               select(all_of(c("time_to_parturition_weeks", "mean_DunedinPACE", "scanning_site"))) |>
               drop_na()
data_epi_dunedinpace <- data_epi_dunedinpace[!duplicated(data_epi_dunedinpace), ]

fit_epi_dunedinpace <- gam(mean_DunedinPACE ~ s(time_to_parturition_weeks, bs = "tp"),
               data = data_epi_dunedinpace,
               method = "REML",
               family = "gaussian")

summ_fit_epi_dunedinpace <- summary(fit_epi_dunedinpace)
gam_fitted_metrics_s01 <- gam_fitted_metrics_s01 |>
    add_row(metric = "DunedinPACE",
            smooth.edf = summ_fit_epi_dunedinpace$s.table[1, "edf"],
            smooth.ref.df = summ_fit_epi_dunedinpace$s.table[1, "Ref.df"],
            smooth.F = summ_fit_epi_dunedinpace$s.table[1, "F"],
            smooth.p = summ_fit_epi_dunedinpace$s.table[1, "p-value"],
            r_sq = summ_fit_epi_dunedinpace$r.sq,
            n = summ_fit_epi_dunedinpace$n
            )

In [ ]:
gam_fitted_metrics_s01_table <- gam_fitted_metrics_s01 |>
    flextable() |>
    width(width = 2.5, j = 1) |>
    width(width = 1.6, j = 2:4) |>
    width(width = 1.2, j = 4) |>
    width(width = 0.8, j = 5) |>
    set_formatter(
        "smooth.edf" = function(x) sprintf("%.2f", x),
        "smooth.ref.df" = function(x) sprintf("%.2f", x),
        "smooth.F" = function(x) sprintf("%.3f", x),
        "smooth.p" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x)),
        "r_sq" = function(x) sprintf("%.3f", x),
        "n" = function(x) sprintf("%d", x)
        ) |>
    set_header_labels(
        metric = "Metric",
        smooth.edf = "Temporal Smooth\nedf",
        smooth.ref.df = "Temporal Smooth\nRef.df",
        smooth.F = "Temporal Smooth\nF-statistic",
        smooth.p = "Temporal Smooth\np-value",
        r_sq = "Model R²(Adjusted)",
        n = "N"
    )
        
read_docx() |>
    body_add_flextable(gam_fitted_metrics_s01_table) |>
    print(target = "tables/smtable_17_gam_fitted_metrics_s01.docx")

## Supplementary Table 1: Participant distribution and session-level variables

In [ ]:
library(arrow)
library(dplyr)
library(stringr)
library(flextable)
library(officer)
library(gtsummary)

In [ ]:
cohort_demographic_long_data <- read_feather("data/cohort_demographic_long_data.feather")

In [ ]:
# Barcelona site 
custom_order <- c("group_gest", "gestational_weeks", "gestational_weeks_gest",
                  "gestational_weeks_non_gest", "postpartum_weeks", "postpartum_weeks_gest",
                  "postpartum_weeks_non_gest")
custom_labels <- c("Group", "Gestational weeks", "Gestational weeks [Gestational Mothers]", "Gestational weeks [Non-Gestational Mothers]",
                   "Time from parturition [weeks]", "Time from parturition [weeks] [Gestational Mothers]", "Time from parturition [weeks] [Non-Gestational Mothers]")

# >>> Create the table
tbl_long_barcelona <- cohort_demographic_long_data %>% 
  filter(acquisition_site == "Barcelona") %>%
  select(all_of(custom_order), session) %>%  
  tbl_summary(
    by = session,
    type = list(where(is.numeric) ~ "continuous", where(is.factor) ~ "categorical"),
    statistic = list(all_continuous() ~ "{mean} ({sd})",
                     all_categorical() ~ "{n} ({p}%)"),
    digits = list(all_continuous() ~ 1, all_categorical() ~ 0),
    label = as.list(setNames(custom_labels, custom_order)),
    missing = 'no'
  ) %>%
  add_n(include=c("group_gest")) %>%
  modify_header(
    stat_1 ~ "ses-1",
    stat_2 ~ "ses-2",
    stat_3 ~ "ses-3",
    stat_4 ~ "ses-4",
    stat_5 ~ "ses-5",
    stat_6 ~ "ses-6"
  ) %>%
  modify_table_body(
    ~ .x %>%
      mutate(
        across(c(label, starts_with("stat_")), ~ str_replace_all(., "0 \\(NA%\\)", "")),
        across(c(label, starts_with("stat_")), ~ str_replace_all(., "NA \\(NA\\)", "")),
      )
  )  
  
ft_barcelona <- as_flex_table(tbl_long_barcelona)


In [ ]:
# Madrid site
custom_order <- c("group_gest", "gestational_weeks", "postpartum_weeks")
custom_labels <- c("Group", "Gestational weeks [Gestational Mothers]", "Time from parturition [weeks] [Gestational Mothers]")

# >>> Create the table
tbl_long_madrid <- cohort_demographic_long_data %>% 
  filter(acquisition_site == "Madrid") %>%
  mutate(
    group_gest = factor(group_gest, levels=c("Gestational Mothers", "Nulliparous Women")),
    session = factor(session, levels=c("ses-3", "ses-4", "ses-6"))
    ) %>%
  select(all_of(custom_order), session) %>%  
  tbl_summary(
    by = session,
    type = list(where(is.numeric) ~ "continuous", where(is.factor) ~ "categorical"),
    statistic = list(all_continuous() ~ "{mean} ({sd})",
                     all_categorical() ~ "{n} ({p}%)"),
    digits = list(all_continuous() ~ 1, all_categorical() ~ 0),                 
    label = as.list(setNames(custom_labels, custom_order)),
    missing = 'no'
  ) %>%
  add_n() %>%
  modify_header(
    stat_1 ~ "ses-3",
    stat_2 ~ "ses-4",
    stat_3 ~ "ses-6",
  ) %>%
  modify_table_body(
    ~ .x %>%
      mutate(
        # Clean up unwanted patterns in the whole table
        across(c(label, starts_with("stat_")), ~ str_replace_all(., "0 \\(NA%\\)", "")),
        across(c(label, starts_with("stat_")), ~ str_replace_all(., "NA \\(NA\\)", "")),
      )
  )  
ft_madrid <- as_flex_table(tbl_long_madrid)

In [ ]:
# write tables
doc <- read_docx() %>%
  body_add_par("Barcelona Cohort", style = "heading 1") %>%
  body_add_flextable(ft_barcelona) %>%
  body_add_par("Madrid Cohort", style = "heading 1") %>%
  body_add_flextable(ft_madrid)
print(doc, target = "tables/smtable_01_participant_per_session_information.docx")

## Supplementary Table 2: General demographic characteristics of the full cohort dataset

In [ ]:
library(arrow)
library(dplyr)
library(stringr)
library(gtsummary)
library(flextable)
library(officer)

In [ ]:
cohort_demographic_data <- read_feather("data/cohort_demographic_data.feather")

In [ ]:
custom_order <- c("acquisition_site",
                  "age_at_ses4",
                  "wais_ses4", 
                  "education_level",
                  "income_monthly")

custom_labels <- c("Acquisition Site [nº of participants (%)]",
                   "Age at ses-4 [years]",
                   "Estimated IQ at ses-4 (WAIS-IV Digits Span)",
                   "Education [nº of participants (%)]",
                   "Monthly Income [euros/month] [nº of participants (%)]"
                   )

p_tests <- c("acquisition_site",
             "age_at_ses4",
             "wais_ses4", 
             "education_level", 
             "income_monthly"
             )

tbl_demographic <- cohort_demographic_data %>%
  select(all_of(custom_order), group_gest) %>%  
  tbl_summary(
    by = group_gest,
    type = list(where(is.numeric) ~ "continuous", where(is.factor) ~ "categorical"),
    statistic = list(all_continuous() ~ "{mean} ({sd})",
                     all_categorical() ~ "{n} ({p}%)"),
    label = as.list(setNames(custom_labels, custom_order)) # ,
    # missing = 'no'
  ) %>%
  add_p(include = p_tests,
        test.args = income_monthly ~ list(workspace=2e9)) %>%
  modify_header(
    stat_1 ~ "Gestational Mother",
    stat_2 ~ "Non-Gestational Mother",
    stat_3 ~ "Nulliparous Women"
  )

In [ ]:
ft <- as_flex_table(tbl_demographic)
doc <- read_docx() %>%
  body_add_flextable(ft)
print(doc, target = "tables/smtable_02_general_demographics_cohort.docx")

## Supplementary Table 3: Perinatal demographics of gestational mothers in the cohort dataset

In [ ]:
library(arrow)
library(dplyr)
library(stringr)
library(gtsummary)
library(flextable)
library(officer)


In [ ]:
cohort_demographic_data <- read_feather("data/cohort_demographic_data.feather") 
cohort_demographic_data_gest <- cohort_demographic_data |>
    filter(group_gest == "gestational_mother")

In [ ]:
custom_order <- c("conception_assisted", "conception_assisted_method",
                  "gestational_weeks_delivery","birth_type",
                  "baby_feeding_type_ses4"
                  )

custom_labels <- c("Assisted reproduction [nº of participants (%)]", "Assisted reproduction method [nº of participants (%)]",
                   "Gestational weeks at delivery [weeks]","Type of parturition [nº of participants (%)]",
                   "Type of Baby Feeding at ses-4 [nº of participants (%)]"
                   )

# >>> Create the table
tbl_demographic_extra_gest <- cohort_demographic_data_gest %>%
  select(all_of(custom_order), group_gest) %>%  
  tbl_summary(
    by = group_gest,
    type = list(where(is.numeric) ~ "continuous", where(is.factor) ~ "categorical"),
    statistic = list(all_continuous() ~ "{mean} ({sd})",
                     all_categorical() ~ "{n} ({p}%)"),
    label = as.list(setNames(custom_labels, custom_order)) # ,
    # missing = 'no'
  ) %>%
  modify_header(
    stat_1 ~ "Gestational Mother",
  )

In [ ]:
ft <- as_flex_table(tbl_demographic_extra_gest)
doc <- read_docx() %>%
  body_add_flextable(ft)
print(doc, target = "tables/smtable_03_perinatal_demographics_gestational_mothers.docx")

## Supplementary Table 8: LMEM fit for brainage gap using brainageR grouping by parturition type on the cohort dataset

In [ ]:
library(arrow)
library(dplyr)
library(tidyr)
library(mgcv)
library(emmeans)

# This script contains the function to format tables with the results
source("code/formatting_functions.R")

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

cohort_data_brainageR <- cohort_data |>
                      select(all_of(c("uncorrected_brainage_gap", "session", "group_gest", "birth_type",
                                      "brainage_model", "age", "scanning_site", "participant_id"))) |>
                      filter(brainage_model == "brainageR") |>
                      mutate(age_centered = age - mean(age)) |> 
                      select(-age) |>
                      rename(age = age_centered)


cohort_data_brainageR <- cohort_data_brainageR |> 
     mutate(birth_type_aux = case_when(
        group_gest != "gestational_mother" ~ "control",
        group_gest == "gestational_mother" ~ birth_type
     )) |>
     mutate(birth_type = factor(birth_type_aux, levels = c("vaginal", "c-section emergency",
                                                           "c-section scheduled", "control"))) |> 
     select(-birth_type_aux, -group_gest)


In [ ]:
lme_fit_birth_type <- gam(uncorrected_brainage_gap ~ birth_type*session +
                                                     s(participant_id, bs='re') + 
                                                    age*scanning_site,
                          data = cohort_data_brainageR, 
                          method = "REML", family = "gaussian")

summ_lme_fit_birth_type <- summary(lme_fit_birth_type)

lme_fit_birth_type_emmeans <- emmeans(lme_fit_birth_type, ~ session | birth_type) 
    
lme_fit_birth_type_emmeans_pairs <- lme_fit_birth_type_emmeans |>
      pairs(reverse = TRUE)

lme_fit_birth_type_emmeans_pairs_tests <- emmeans::test(lme_fit_birth_type_emmeans_pairs, by = NULL, adjust = 'none') |> 
      relocate(birth_type, .before = contrast) |>
      filter(str_detect(contrast, paste(c(str_escape("- (ses-1)"), str_escape("- (ses-3)")), collapse = "|"))) |>
      mutate(sesref = str_extract(contrast, "ses-\\d+\\)$")) |> # For ordering the contrasts
      arrange(birth_type, sesref, contrast) |>
      select(-sesref)
lme_fit_birth_type_emmeans_pairs_tests$pFDR <- p.adjust(lme_fit_birth_type_emmeans_pairs_tests$p.value, method = "fdr") 
colnames(lme_fit_birth_type_emmeans_pairs_tests) <- c("Birth Type", "Contrast", "Estimate", "SE",
                                                      "df", "t.ratio", "p.value", "pFDR")

In [ ]:
birth_type_table <- create_lme_fit_tables_docx(
    lme_fit = lme_fit_birth_type,
    em_comparisons = lme_fit_birth_type_emmeans_pairs_tests,
    ptable_substitutions = substitutions_ptable_lme_fit_birth_type,
    stable_substitutions = substitutions_stable_lme_fit_birth_type,
    emtable_substitutions = substitutions_emtable_test_birth_type,
    type_dataset = "cohort_birth_type",
    docx_filename = "tables/smtable_08_A_lme_fit_brainageR_cohort_birth_type.docx"
)

In [ ]:
# Adding between group cross-sectional comparisons for each session 
lme_fit_birth_type_emmeans_ses <- emmeans(lme_fit_birth_type, ~ birth_type | session)
lme_fit_birth_type_emmeans_ses_pairs <- pairs(lme_fit_birth_type_emmeans_ses)
lme_fit_birth_type_emmeans_ses_pairs_tests <- emmeans::test(lme_fit_birth_type_emmeans_ses_pairs, by = NULL, adjust = "none") |>
    relocate(session, .before = contrast)
lme_fit_birth_type_emmeans_ses_pairs_tests$pFDR <- p.adjust(lme_fit_birth_type_emmeans_ses_pairs_tests$p.value, method = "fdr") 
colnames(lme_fit_birth_type_emmeans_ses_pairs_tests) <- c("Session", "Contrast", "Estimate", "SE",
                                                          "df", "t.ratio", "p.value", "pFDR")

# Adding between group longitudinal comparisons (just keeping the ones of interest)
lme_fit_birth_type_emmeans_long <- contrast(lme_fit_birth_type_emmeans_pairs, "pairwise", by = NULL, adjust = 'none')
lme_fit_birth_type_emmeans_long_tests <- emmeans::test(lme_fit_birth_type_emmeans_long, 
                                                       by = NULL, adjust = "none") |>
  mutate(ses_int1 = str_extract(contrast, "(\\(ses-[1-6]\\) - \\(ses-[1-6]\\))"),
         ses_int2 = str_remove(str_extract(contrast, "- \\((\\(ses-[1-6]\\) - \\(ses-[1-6]\\))"), "- \\("),
         contrast_aux = str_remove_all(contrast, "\\((\\(ses-[1-6]\\) - \\(ses-[1-6]\\))")) |>
  filter(ses_int1 == ses_int2) |>
  filter(str_detect(ses_int1, paste(c(str_escape("- (ses-1)"), str_escape("- (ses-3)")), collapse = "|"))) |>   
  mutate(sesref = str_extract(ses_int1, "ses-\\d+\\)$")) |> # For ordering the contrasts
  arrange(sesref, contrast) |>
  select(-ses_int2, -sesref) |>
  relocate(ses_int1, .before = contrast) |>
  relocate(contrast_aux, .before = contrast) |>
  select(-contrast)
lme_fit_birth_type_emmeans_long_tests$pFDR <- p.adjust(lme_fit_birth_type_emmeans_long_tests$p.value, method = "fdr")
colnames(lme_fit_birth_type_emmeans_long_tests) <- c("Sessions", "Contrast", "Estimate", "SE",
                                                     "df", "t.ratio", "p.value", "pFDR")

In [ ]:
birth_aux_type_table <- create_lme_fit_additional_em_tables_docx(
    em_comparisons_cross = lme_fit_birth_type_emmeans_ses_pairs_tests,
    em_comparisons_long = lme_fit_birth_type_emmeans_long_tests,
    em_cross_table_substitutions = substitutions_em_ses_table_birth_type,
    em_long_table_substitutions = substitutions_em_long_table_birth_type,
    docx_filename = "tables/smtable_08_B_additional_comparisons_brainageR_cohort_birth_type.docx")

## Supplementary Table 12: LMEM fit for brainage gap using pyment on the cohort dataset

In [ ]:
library(arrow)

# This script contains the function to format tables with the results
source("code/formatting_functions.R")
# This script contains the function to fit the lme model for each brainage model and create the table with the emmens comparisons
source("code/lme_fit_per_brainage_model_cohort.R")

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

lme_fit_pyment <- lme_fit_per_brainage_model_cohort(data = cohort_data,
                                                    abrainage_model = "pyment")

In [ ]:
lme_tables <- create_lme_fit_tables_docx(lme_fit = lme_fit_pyment$lme_fit,
                                         em_comparisons = lme_fit_pyment$emmeans_tests,
                                         ptable_substitutions = substitutions_ptable_lme_fit_cohort,
                                         stable_substitutions = substitutions_stable_lme_fit_cohort,
                                         emtable_substitutions = substitutions_emtable_test_cohort,
                                         type_dataset = "cohort",
                                         docx_filename = "tables/smtable_12_lme_fit_pyment_cohort.docx")

## Supplementary Table 13: LMEM fit for brainage gap using DeepBrainNet on the cohort dataset

In [ ]:
library(arrow)

# This script contains the function to format tables with the results
source("code/formatting_functions.R")
# This script contains the function to fit the lme model for each brainage model and create the table with the emmens comparisons
source("code/lme_fit_per_brainage_model_cohort.R")

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

lme_fit_deepbrainnet <- lme_fit_per_brainage_model_cohort(data = cohort_data,
                                                          abrainage_model = "DeepBrainNet")

In [ ]:
lme_tables <- create_lme_fit_tables_docx(lme_fit = lme_fit_deepbrainnet$lme_fit,
                                         em_comparisons = lme_fit_deepbrainnet$emmeans_tests,
                                         ptable_substitutions = substitutions_ptable_lme_fit_cohort,
                                         stable_substitutions = substitutions_stable_lme_fit_cohort,
                                         emtable_substitutions = substitutions_emtable_test_cohort,
                                         type_dataset = "cohort",
                                         docx_filename = "tables/smtable_13_lme_fit_DeepBrainNet_cohort.docx")

## Supplementary Table 14: LMEM fit for brainage gap using brainAGE on the cohort dataset

In [ ]:
library(arrow)

# This script contains the function to format tables with the results
source("code/formatting_functions.R")
# This script contains the function to fit the lme model for each brainage model and create the table with the emmens comparisons
source("code/lme_fit_per_brainage_model_cohort.R")

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

lme_fit_brainAGE <- lme_fit_per_brainage_model_cohort(data = cohort_data,
                                                      abrainage_model = "brainAGE")

In [ ]:
lme_tables <- create_lme_fit_tables_docx(lme_fit = lme_fit_brainAGE$lme_fit,
                                         em_comparisons = lme_fit_brainAGE$emmeans_tests,
                                         ptable_substitutions = substitutions_ptable_lme_fit_cohort,
                                         stable_substitutions = substitutions_stable_lme_fit_cohort,
                                         emtable_substitutions = substitutions_emtable_test_cohort,
                                         type_dataset = "cohort",
                                         docx_filename = "tables/smtable_14_lme_fit_brainAGE_cohort.docx")

## Supplementary Table 16: LMEM fit for brainage gap using pyment, DeepBrainNet and brainAGE on the dense-sampling dataset

In [ ]:
library(arrow)

# This script contains the function to format tables with the results
source("code/formatting_functions.R")
# This script contains the function to fit the lme model for each brainage model and create the table with the emmens comparisons
source("code/lme_fit_per_brainage_model_dense.R")

In [ ]:
dense_data <- read_feather("data/dense_sampling_brainage_data.feather")

# pyment
lme_fit_pyment <- lme_fit_per_brainage_model_dense(data = dense_data,
                                                   abrainage_model = "pyment")

lme_tables_pyment <- create_lme_fit_tables_docx(
    lme_fit = lme_fit_pyment$lme_fit,
    em_comparisons = lme_fit_pyment$emmeans_tests,
    ptable_substitutions = substitutions_ptable_lme_fit_dense,
    stable_substitutions = substitutions_stable_lme_fit_dense,
    emtable_substitutions = substitutions_emtable_test_dense,
    type_dataset = "dense",
    docx_filename = "tables/smtable_16_A_lme_fit_pyment_dense.docx"
)

In [ ]:
# DeepBrainNet
lme_fit_deepbrainnet <- lme_fit_per_brainage_model_dense(data = dense_data,
                                                         abrainage_model = "DeepBrainNet")

lme_tables_deepbrainnet <- create_lme_fit_tables_docx(
    lme_fit = lme_fit_deepbrainnet$lme_fit,
    em_comparisons = lme_fit_deepbrainnet$emmeans_tests,
    ptable_substitutions = substitutions_ptable_lme_fit_dense,
    stable_substitutions = substitutions_stable_lme_fit_dense,
    emtable_substitutions = substitutions_emtable_test_dense,
    type_dataset = "dense",
    docx_filename = "tables/smtable_16_B_lme_fit_deepbrainnet_dense.docx"
)

In [ ]:
# brainAGE
lme_fit_brainAGE <- lme_fit_per_brainage_model_dense(data = dense_data,
                                                     abrainage_model = "brainAGE")

lme_tables_brainAGE <- create_lme_fit_tables_docx(
    lme_fit = lme_fit_brainAGE$lme_fit,
    em_comparisons = lme_fit_brainAGE$emmeans_tests,
    ptable_substitutions = substitutions_ptable_lme_fit_dense,
    stable_substitutions = substitutions_stable_lme_fit_dense,
    emtable_substitutions = substitutions_emtable_test_dense,
    type_dataset = "dense",
    docx_filename = "tables/smtable_16_C_lme_fit_brainAGE_dense.docx"
)

In [ ]:
# Join tables in one
# ptables
ptable_pyment <- lme_tables_pyment$ptable
ptable_deepbrainnet <- lme_tables_deepbrainnet$ptable
ptable_brainAGE <- lme_tables_brainAGE$ptable

ptable_pyment$brainage_model <- "pyment"
ptable_deepbrainnet$brainage_model <- "DeepBrainNet"
ptable_brainAGE$brainage_model <- "brainAGE"

joined_ptables <- bind_rows(ptable_pyment, ptable_deepbrainnet, ptable_brainAGE) |>
                                 relocate(brainage_model, .before = Coefficient)

colnames(joined_ptables)[1] <- "Brainage Model"

joined_ptables_formated <- joined_ptables |> 
  flextable() |> 
  width(width = 1.5, j = 1) |>
  width(width = 3, j = 2) |>
  width(width = 1, j = 3:6) |> 
  set_formatter(
    "Estimate" = function(x) sprintf("%.3f", x),
    "Std.Error" = function(x) sprintf("%.3f", x),
    "t.value" = function(x) sprintf("%.3f", x),
    "p.value" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x))
  )

# stables
stable_pyment <- lme_tables_pyment$stable
stable_deepbrainnet <- lme_tables_deepbrainnet$stable
stable_brainAGE <- lme_tables_brainAGE$stable

stable_pyment$brainage_model <- "pyment"
stable_deepbrainnet$brainage_model <- "DeepBrainNet"
stable_brainAGE$brainage_model <- "brainAGE"

joined_stables <- bind_rows(stable_pyment, stable_deepbrainnet, stable_brainAGE) |>
                                 relocate(brainage_model)

colnames(joined_stables)[1] <- "Brainage Model"

joined_stables_formated <- joined_stables |> 
  flextable() |> 
  width(width = 1.5, j = 1) |>
  width(width = 3, j = 2) |>
  width(width = 1, j = 3:6) |> 
  set_formatter(
    "edf" = function(x) sprintf("%.2f", x),
    "Ref.df" = function(x) sprintf("%.2f", x),
    "F" = function(x) sprintf("%.3f", x),
    "p.value" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x))
  )

# emtables
emtable_pyment <- lme_tables_pyment$emtable
emtable_deepbrainnet <- lme_tables_deepbrainnet$emtable
emtable_brainAGE <- lme_tables_brainAGE$emtable

emtable_pyment$brainage_model <- "pyment"
emtable_deepbrainnet$brainage_model <- "DeepBrainNet"
emtable_brainAGE$brainage_model <- "brainAGE"

joined_emtables <- bind_rows(emtable_pyment, emtable_deepbrainnet, emtable_brainAGE) |>
                                 relocate(brainage_model)

colnames(joined_emtables)[1] <- "Brainage Model"

joined_emtables_formated <- joined_emtables |> 
  flextable() |> 
  width(width = 1, j = 1) |>
  width(width = 1.4, j = 2) |>
  width(width = 0.85, j = 3:8) |> 
  set_formatter(
    "Estimate" = function(x) sprintf("%.3f", x),
    "SE" = function(x) sprintf("%.3f", x),
    "df" = function(x) sprintf("%.2f", x),
    "t.ratio" = function(x) sprintf("%.3f", x),
    "p.value" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x)),
    "pFDR" = function(x) ifelse(x < 0.001, "<0.001", sprintf("%.3f", x))
  )

# fit quality metrics
summ_pyment <- summary(lme_fit_pyment$lme_fit)
summ_deepbrainnet <- summary(lme_fit_deepbrainnet$lme_fit)
summ_brainAGE <- summary(lme_fit_brainAGE$lme_fit)

joined_fit_quality_metrics <- data.frame(
    Brainage_Model = c("pyment", "DeepBrainNet", "brainAGE"),
    r.sq = c(summ_pyment$r.sq, summ_deepbrainnet$r.sq, summ_brainAGE$r.sq),
    dev.expl = c(summ_pyment$dev.expl, summ_deepbrainnet$dev.expl, summ_brainAGE$dev.expl),
    n = c(summ_pyment$n, summ_deepbrainnet$n, summ_brainAGE$n)
)

joined_fit_quality_metrics_formated <- joined_fit_quality_metrics |> 
  flextable() |> 
  width(width = 1.5, j = 1) |>
  width(width = 1.5, j = 2:4) |> 
  set_formatter(
    "r.sq" = function(x) sprintf("%.3f", x),
    "dev.expl" = function(x) sprintf("%.3f", x),
    "n" = function(x) sprintf("%d", x)
  ) |> 
  set_header_labels(
    Brainage_Model = "Brainage Model",
    r.sq = "R-squared",
    dev.expl = "Deviance Explained",
    n = "N"
  )

joined_tables_docx <- read_docx() |>
  body_add_par("Table with LME parametric coefficients") |>
  body_add_flextable(joined_ptables_formated) |>
  body_add_par("Table with LME smooth terms") |>
  body_add_flextable(joined_stables_formated) |>
  body_add_par("Fit quality metrics") |>
  body_add_flextable(joined_fit_quality_metrics_formated) |>
  body_add_par("Table with LME estimated marginal means") |>
  body_add_flextable(joined_emtables_formated) |>
  print(target = "tables/smtable_16_lme_fit_other_brainage_algorithms_joined_dense.docx")

## Supplementary Table 19: Brain-age models performance metrics

In [ ]:
library(arrow)
library(dplyr)
library(tidyr)
library(flextable)
library(officer)

In [ ]:
cohort_data <- read_feather("data/cohort_brainage_data.feather")

# Restricting the analysis to nulliparous women (control)
perf_data <- cohort_data |> filter(group_gest == "nulliparous_women")

perf_metrics <- perf_data |>
    group_by(brainage_model) |>
    summarise(mean_ubag = mean(uncorrected_brainage_gap, na.rm = TRUE),
              sd_ubag = sd(uncorrected_brainage_gap, na.rm = TRUE),
              mae_ubag = mean(abs(uncorrected_brainage_gap), na.rm = TRUE),
              mae_cbag = mean(abs(corrected_brainage_gap_per_dataset), na.rm = TRUE),
              r_uba = cor(uncorrected_brainage_gap + age, age, method = "pearson", use = "complete.obs"),
              r_pad_uba = cor(uncorrected_brainage_gap, age, method = "pearson", use = "complete.obs"),
              r_cbag = cor(corrected_brainage_gap_per_dataset + age, age, method = "pearson", use = "complete.obs"),
              r_pad_cbag = cor(corrected_brainage_gap_per_dataset, age, method = "pearson", use = "complete.obs"),
              n = n())

# Test-retest performance metrics (using ses-3 and ses-4 sessions since they are the closest in time)
trt_data <- perf_data |> 
    filter(session %in% c("ses-3", "ses-4")) |> 
    group_by(participant_id, brainage_model) |> 
    filter(n_distinct(session) == 2) |>
    ungroup()

trt_metrics <- trt_data |> group_by(brainage_model, participant_id) |>
    mutate(adj_mad_ubag = abs(uncorrected_brainage_gap[session == "ses-4"] - uncorrected_brainage_gap[session == "ses-3"]),
           adj_mad_cbag = abs(corrected_brainage_gap_per_dataset[session == "ses-4"] - corrected_brainage_gap_per_dataset[session == "ses-3"]),
           time_diff = age[session == "ses-4"] - age[session == "ses-3"]) |> 
    ungroup(participant_id) |>
    summarise(adj_mad_ubag = mean(adj_mad_ubag, na.rm = TRUE),
              adj_mad_cbag = mean(adj_mad_cbag, na.rm = TRUE),
              mean_time_diff = mean(time_diff, na.rm = TRUE),
              sd_time_diff = sd(time_diff, na.rm = TRUE),
              n = n())

# Merging the performance metrics 
perf_metrics_all <- merge(perf_metrics, trt_metrics, by = "brainage_model")

# Formatting and creating the table

perf_metrics_formatted <- perf_metrics_all  |>
    select(
        brainage_model, mean_ubag, sd_ubag,
        mae_ubag, r_uba, r_pad_uba, adj_mad_ubag
    ) |> 
    flextable() |>
    width(width = 1.5, j = 1) |>
    width(width = 0.7, j = 2:7) |>
    set_formatter(
        "mean_ubag" = function(x) sprintf("%.3f", x),
        "sd_ubag" = function(x) sprintf("%.3f", x),
        "mae_ubag" = function(x) sprintf("%.3f", x),
        "r_uba" = function(x) sprintf("%.3f", x),
        "r_pad_uba" = function(x) sprintf("%.3f", x),
        "adj_mad_ubag" = function(x) sprintf("%.3f", x)
    ) |>
    set_header_labels(
        brainage_model = "Brainage Model",
        mean_ubag = "Mean BAG \n(years)",
        sd_ubag = "SD BAG \n(years)",
        mae_ubag = "MAE \n(years)",
        r_uba = "r",
        r_pad_uba = "r BAG",
        adj_mad_ubag = "test-retest Adj MAD \n (years)"
    )

perf_metrics_docx <- read_docx() |>
    body_add_flextable(perf_metrics_formatted) |>
    body_add_par(sprintf("Performance metrics computed in Nsessions = %d; 
                          test-retest performance metrics computed in Nparticipants = %d;
                          time elapsed between sessions = %.3f ± %.3f years",
                          perf_metrics$n[1], trt_metrics$n[1], trt_metrics$mean_time_diff[1], trt_metrics$sd_time_diff[1])) |>
    print(target = "tables/smtable_19_performance_metrics_brainage_models.docx")
